# Cell 1 — Temporal validation and model selection (Notebook 3)

**Purpose:** compare M0 majority baseline, M1 Logistic Regression URL, M2 Logistic
Regression URL+HTML, M3 Random Forest URL, and M4 Random Forest URL+HTML.
Use the frozen inclusive **2025-07-09** cutoff: earlier official-training pages
fit candidates; later official-training pages select settings by **Average
Precision (AP)**. These are development results, not final test performance.

**Test embargo:** this notebook never opens the official test dataset, its schema,
or its predictions. Do not attach raw HTML. Add the saved **FULL Notebook 2 output**
containing the training feature Parquet and three required metadata files.
Flat and nested input layouts are supported automatically.

**Run:** CPU, start with `RUN_MODE = "SMOKE"` in Cell 2, Run All, and check
`SMOKE PASS`. Then change only RUN_MODE to `"FULL"`, restart the session, and Run All.
Save a Kaggle version **with outputs** after `FULL PASS`. Internet is unnecessary
when dependencies are installed. If an import fails, install only the named missing
package using a temporary console/session command, restart, and rerun; do not
blindly upgrade the environment or add a sixteenth cell.

Frozen grids: LR C = [0.1, 1.0, 10.0]; RF max_depth = [None, 20], 300 trees.
Exact AP ties keep the first predeclared candidate. No threshold tuning,
class weighting, feature selection, or split changes. Predict uses estimator
`predict()` (class 0 wins exact probability ties); the nominal threshold is 0.5.
Final M0–M4 are refitted on **all official training rows in FULL**.
SMOKE uses a stratified sample of both frozen partitions and cannot authorize NB4.
All-missing fit columns stop execution rather than silently dropping predictors.
RF fits are sequential; resource failure means STOP, never a smaller hidden grid.


In [ ]:
# Cell 2 — Settings, dependencies, verified FULL NB2 inputs, training-only load
RUN_MODE = "SMOKE"                         # Change to FULL only after SMOKE PASS.
MAKE_ZIP = True
SMOKE_TRAIN_ROWS, SMOKE_VALIDATION_ROWS = 2000, 1000
INPUT_ROOT = "/kaggle/input"
OUTPUT_ROOT = "/kaggle/working/nb03"

import os, sys, json, hashlib, time, gc, warnings, shutil, platform, zipfile
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
try:
    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    import joblib, sklearn, pyarrow, openpyxl, psutil
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.dummy import DummyClassifier
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
        f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
        confusion_matrix, precision_recall_curve, roc_curve)
except ImportError as exc:
    raise RuntimeError(f"Missing dependency: {exc}. Install that package, restart, Run All.") from exc

assert RUN_MODE in {"SMOKE", "FULL"}, "RUN_MODE must be SMOKE or FULL"
SEED = 42
np.random.seed(SEED)
STARTED = time.perf_counter()
RUN_ID = datetime.now(timezone.utc).isoformat()
OUT = Path(OUTPUT_ROOT) / RUN_MODE.lower()
for name in ["models", "tables", "manifests", "logs", "figures"]:
    (OUT / name).mkdir(parents=True, exist_ok=True)

def write_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    # Imputer missing_values=np.nan is a setting, not a numeric metric.
    # Encode nonfinite settings explicitly instead of emitting invalid JSON NaN.
    def safe(v):
        if isinstance(v, np.generic): v = v.item()
        if isinstance(v, dict): return {str(k): safe(x) for k, x in v.items()}
        if isinstance(v, (list, tuple)): return [safe(x) for x in v]
        if isinstance(v, float) and not np.isfinite(v): return str(v)
        return v
    temp.write_text(json.dumps(safe(value), indent=2, allow_nan=False,
        default=str), encoding="utf-8")
    temp.replace(path)

def require(ok, message):
    if not bool(ok):
        raise RuntimeError("STOP: " + message)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def file_record(path):
    return {"path": str(path), "bytes": Path(path).stat().st_size, "sha256": sha256_file(path)}

# Invalidate any previous successful run before doing new work.
write_json(OUT / "manifests/manifest_nb03.json", {
    "run_id": RUN_ID, "mode": RUN_MODE, "ready_for_nb04": False,
    "config_frozen": False, "official_test_loaded": False,
    "official_test_evaluated": False, "blockers": ["Run incomplete; reach Cell 15."]})
DATASET = {"id": "phreshphish/phreshphish", "version": "1.0.1",
           "revision": "eabec4b7a66324b79cc8a0ad856d1731dc26fe1a"}
CUTOFF = "2025-07-09"
EXPECTED_COUNTS = {
    "internal_train": {"rows": 397480, "benign": 204388, "phishing": 193092,
                       "start": "2024-07-02", "end": "2025-07-09"},
    "internal_validation": {"rows": 100775, "benign": 72341, "phishing": 28434,
                            "start": "2025-07-10", "end": "2025-09-08"}}
EXPECTED_ROWS = 498255
EXPECTED_HASHES = {
    "hybrid_features_train.parquet": "e0a5768e7b60916db32f57f0c078978017d88731f2baf3121ebfe77fbdd3030f",
    "final_feature_set.json": "3bb24f1e9c1ffc6a02d0a7307e60fa78c68fc0f1dedb3309d0ab025cd97d729a",
    "html_feature_dictionary.csv": "76b40cc8c1b939353b69b4859a2617f76bea21e79af17ec157135c5a1c464a6d"}

# Only NB2 manifest basenames are opened during discovery; no test file is opened.
root = Path(INPUT_ROOT)
eligible, rejected = [], []
for path in sorted(root.rglob("manifest_nb02.json")):
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
        if obj.get("mode") == "FULL" and obj.get("ready_for_nb03") is True and obj.get("blockers") == []:
            eligible.append((path, obj))
        else:
            rejected.append(str(path))
    except (ValueError, OSError):
        rejected.append(str(path))
require(len(eligible) == 1, f"Need exactly one ready FULL manifest_nb02.json; found {len(eligible)}. "
        f"Eligible: {[str(x[0]) for x in eligible]}; rejected: {rejected}. "
        "Attach saved FULL NB2 output (unpacked files), remove duplicate mounts. No raw HTML needed.")
manifest_path, nb2 = eligible[0]
# New and legacy Kaggle paths both put the dataset or notebook attachment under
# the first input component; restrict discovery to that relevant mount.
relative_parts = manifest_path.relative_to(root).parts
mount_parts = 3 if relative_parts[0] in {"datasets", "notebooks"} and len(relative_parts) >= 4 else 1
mount = root.joinpath(*relative_parts[:mount_parts])
if mount.is_file():
    mount = root
inputs = {"manifest_nb02.json": manifest_path}
for basename in EXPECTED_HASHES:
    matches = sorted(mount.rglob(basename))
    require(len(matches) == 1, f"Missing/ambiguous {basename} within {mount}: {matches}. "
            "Attach one NB2 output or a private training-only dataset with all four required files.")
    inputs[basename] = matches[0]
input_records = {name: file_record(path) for name, path in inputs.items()}
for name, expected in EXPECTED_HASHES.items():
    require(input_records[name]["sha256"] == expected, f"Checksum mismatch for {inputs[name]}. "
            "Use the exact verified FULL NB2 output; do not edit checksums to bypass this check.")
require(nb2.get("official_test_evaluated") is False, "NB2 must report official_test_evaluated=false")
# Accommodate nested NB2 provenance without assuming undocumented wrapper keys.
def walk_values(obj):
    if isinstance(obj, dict):
        for value in obj.values():
            yield from walk_values(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk_values(value)
    else:
        yield obj
provenance = list(walk_values(nb2))
for value in DATASET.values():
    require(any(str(value) in str(v) for v in provenance), f"NB2 provenance missing {value}")
require("artifacts" in nb2, "NB2 manifest has no artifacts")
def matching_artifacts(obj, basename, context=""):
    hits = []
    if isinstance(obj, dict):
        direct = [str(v) for v in obj.values() if isinstance(v, str)]
        if basename in context or any(basename in v for v in direct):
            hits.append(obj)
        for k, v in obj.items():
            hits.extend(matching_artifacts(v, basename, str(k)))
    elif isinstance(obj, list):
        for v in obj:
            hits.extend(matching_artifacts(v, basename, context))
    return hits
for name in EXPECTED_HASHES:
    hits = matching_artifacts(nb2["artifacts"], name)
    require(any(input_records[name]["sha256"] in list(walk_values(h)) for h in hits),
            f"NB2 artifacts missing matching hash for {name}")
    for hit in hits:
        for key in ["size_bytes", "bytes"]:
            if key in hit:
                require(int(hit[key]) == input_records[name]["bytes"], f"NB2 size mismatch: {name}")
fs = json.loads(inputs["final_feature_set.json"].read_text(encoding="utf-8"))
require(fs.get("mode") == "FULL" and fs.get("official_test_evaluated") is False,
        "Final feature JSON is not embargo-safe FULL")
require(str(fs.get("internal_cutoff_inclusive"))[:10] == CUTOFF, "Feature JSON cutoff mismatch")
require(fs.get("decision_partition") == "internal_train", "Feature decisions must use internal_train")
require(fs.get("removed_h_features") == [], "Frozen handoff retained all 10 HTML features")
for key in ["url_features_unchanged", "h_ranges_valid", "unique_sha"]:
    def claims(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == key: yield v
                yield from claims(v)
        elif isinstance(obj, list):
            for v in obj: yield from claims(v)
    require(all(v is True for v in claims(nb2)), f"NB2 has a failed {key} claim")

free_gb = shutil.disk_usage(OUT).free / 2**30
available_gb = psutil.virtual_memory().available / 2**30
print(f"Mode={RUN_MODE}; available RAM={available_gb:.2f} GiB; free disk={free_gb:.2f} GiB")
require(free_gb >= (3 if RUN_MODE == "FULL" else .25), "Insufficient output disk; free space and restart")
require(available_gb >= (4 if RUN_MODE == "FULL" else .25), "Insufficient available RAM; restart/free memory")
print("These are minimum startup checks, not a guarantee that a 300-tree forest will fit.")
for name, path in inputs.items(): print(name, "->", path)
# The ONLY source data read in this notebook is the official-training feature file.
df = pd.read_parquet(inputs["hybrid_features_train.parquet"]).reset_index(drop=True)
require(len(df) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS} training rows; got {len(df)}")
packages = {p: version(p) for p in ["numpy", "pandas", "scikit-learn", "scipy", "joblib",
            "pyarrow", "matplotlib", "openpyxl", "psutil"]}
packages["python"] = platform.python_version()
package_table = pd.DataFrame(packages.items(), columns=["package", "version"])
package_table.to_csv(OUT / "manifests/package_versions_nb03.csv", index=False)
blockers, runtime_rows, candidate_rows, chosen, validation_predictions = [], [], [], {}, {}
print(f"PASS: verified FULL NB2 training inputs; loaded {len(df):,} official-training rows only.")


In [ ]:
# Cell 3 — Frozen feature order, numeric schema, no diagnostic predictors
EXPECTED_U = ["url_length", "hostname_length", "path_length", "query_length",
 "num_dots", "num_hyphens", "num_digits", "digit_ratio", "num_subdomains",
 "path_depth", "num_query_params", "has_ip_host", "has_punycode", "has_at_symbol", "url_entropy"]
EXPECTED_H = ["anchor_external_ratio", "form_external_ratio", "form_invalid_action_ratio",
 "script_external_ratio", "image_external_ratio", "iframe_external_ratio",
 "link_resource_external_ratio", "overall_external_resource_ratio",
 "unique_external_registered_domains", "dominant_resource_domain_mismatch"]
U, H = fs["url_only_features"], fs["retained_h_features"]
F = fs["combined_features"]
require(U == EXPECTED_U and H == EXPECTED_H and F == U + H, "Frozen feature names/order mismatch")
require(fs["url_features"] == U and fs["candidate_h_features"] == H, "Feature JSON lists disagree")
META = ["sha256", "url", "label", "date", "source_row", "record_id", "development_partition"]
require(df.columns.is_unique, "Duplicate dataframe column names")
require(set(META + F).issubset(df.columns), f"Missing columns: {set(META + F) - set(df.columns)}")
require(not df[META].isna().any().any(), "Mandatory metadata contains missing values")
require(df["sha256"].is_unique, "Training SHA256 identifiers are not unique")
require(set(df["label"].unique()) == {0, 1}, "Labels must be exactly 0 and 1")
require(all(pd.api.types.is_numeric_dtype(df[f]) for f in F), "Predictors must be numeric")
require(not np.isinf(df[F].to_numpy(dtype=float)).any(), "Infinite predictor values: STOP, investigate upstream")
for f in H:
    values = df[f].dropna()
    if f == "unique_external_registered_domains":
        require(((values >= 0) & (values % 1 == 0)).all(), f"Invalid count: {f}")
    else:
        require(values.between(0, 1).all(), f"Invalid range: {f}")
for key in ["url_only_features", "combined_features", "retained_h_features"]:
    if key in nb2:
        require(nb2[key] == fs[key], f"NB2/feature manifest disagree: {key}")
FEATURES = {"M0": ["dummy_constant"], "M1": U, "M2": F, "M3": U, "M4": F}
MODEL_NAMES = {"M0": "Majority reference", "M1": "LR URL", "M2": "LR URL+HTML",
               "M3": "RF URL", "M4": "RF URL+HTML"}
feature_table = pd.DataFrame({"order": range(1, 26), "feature": F,
                              "family": ["URL"] * 15 + ["HTML"] * 10})
feature_table.to_csv(OUT / "tables/frozen_features.csv", index=False)
print("PASS: 15 URL + 10 retained HTML features; metadata/diagnostics excluded by explicit allowlist.")


In [ ]:
# Cell 4 — Reuse frozen temporal assignment; FULL counts checked even in SMOKE
DATES = pd.to_datetime(df["date"], errors="raise", utc=True).dt.tz_convert(None).dt.normalize()
require(not DATES.isna().any(), "Unparseable/missing dates")
expected_assignment = np.where(DATES <= pd.Timestamp(CUTOFF), "internal_train", "internal_validation")
require(np.array_equal(df["development_partition"].to_numpy(), expected_assignment),
        "Stored development_partition disagrees with frozen inclusive cutoff")
indices = {part: np.flatnonzero(expected_assignment == part) for part in EXPECTED_COUNTS}
require(DATES.iloc[indices["internal_train"]].max() < DATES.iloc[indices["internal_validation"]].min(),
        "Chronological partitions overlap")
summary_rows = []
for part, ix in indices.items():
    y = df.iloc[ix]["label"]
    row = {"partition": part, "rows": len(ix), "benign": int((y == 0).sum()),
           "phishing": int((y == 1).sum()), "start": str(DATES.iloc[ix].min().date()),
           "end": str(DATES.iloc[ix].max().date()), "prevalence": float(y.mean())}
    for k, v in EXPECTED_COUNTS[part].items():
        require(row[k] == v, f"Frozen {part} {k}: expected {v}, got {row[k]}")
    summary_rows.append(row)
require(not np.intersect1d(indices["internal_train"], indices["internal_validation"]).size,
        "Partitions overlap")
split_table = pd.DataFrame(summary_rows)
split_table.to_csv(OUT / "tables/temporal_split_summary.csv", index=False)
# Proportionate sampling within each existing partition is a SMOKE code check,
# never a new random development split. Ensure both labels are represented.
def smoke_indices(ix, n):
    if len(ix) <= n: return ix.copy()
    rng = np.random.default_rng(SEED)
    labels = df.iloc[ix]["label"].to_numpy()
    n1 = min(max(1, int(round(n * labels.mean()))), n - 1)
    picked = np.concatenate([rng.choice(ix[labels == 0], n - n1, replace=False),
                             rng.choice(ix[labels == 1], n1, replace=False)])
    return np.sort(picked)
train_ix = indices["internal_train"] if RUN_MODE == "FULL" else smoke_indices(indices["internal_train"], SMOKE_TRAIN_ROWS)
val_ix = indices["internal_validation"] if RUN_MODE == "FULL" else smoke_indices(indices["internal_validation"], SMOKE_VALIDATION_ROWS)
refit_ix = np.arange(len(df)) if RUN_MODE == "FULL" else np.sort(np.concatenate([train_ix, val_ix]))
y_train = df.iloc[train_ix]["label"].to_numpy(dtype=int)
y_val = df.iloc[val_ix]["label"].to_numpy(dtype=int)
y_refit = df.iloc[refit_ix]["label"].to_numpy(dtype=int)
for y in [y_train, y_val, y_refit]: require(set(y) == {0, 1}, "Both classes required in each execution sample")
require(not df.iloc[train_ix][F].isna().all().any(),
        "All-missing internal-fit predictor; STOP rather than silently dropping a feature. In SMOKE increase sample size.")
split_info = {"cutoff_inclusive": CUTOFF, "rule": "whole day <= cutoff: internal_train; later: internal_validation",
    "original_rule": "whole-day cumulative count nearest 80%; earlier tie wins; frozen by NB1",
    "development_partition_agreement": True, "sha_disjoint": True, "chronological": True,
    "official_training_rows": len(df), "partitions": summary_rows,
    "execution_rows": {"internal_train": len(train_ix), "internal_validation": len(val_ix), "refit": len(refit_ix)},
    "mode": RUN_MODE, "global_strict_temporal_boundary": False,
    "official_boundary_caveat": "Official benign train/test share 2025-09-08 (handoff provenance only; test not read)."}
write_json(OUT / "manifests/temporal_split.json", split_info)
print(split_table.to_string(index=False))
print("Execution sample:", split_info["execution_rows"])
if RUN_MODE == "SMOKE": print("PROVISIONAL: sampled forests validate code only; NB4 readiness stays false.")


In [ ]:
# Cell 5 — Full official-training chronology and prevalence diagnostics
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
COLORS = ["#777777", "#0072B2", "#56B4E9", "#D55E00", "#009E73"]
figure_paths = []
def save_figure(fig, filename):
    fig.tight_layout()
    path = OUT / "figures" / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    figure_paths.append(str(path.relative_to(OUT)))
fig, ax = plt.subplots(figsize=(9, 3.2))
for i, row in split_table.iterrows():
    ax.plot(pd.to_datetime([row["start"], row["end"]]), [i, i], linewidth=12, color=COLORS[i+1], solid_capstyle="butt")
ax.axvline(pd.Timestamp(CUTOFF), color="black", linestyle="--", label="Frozen cutoff: 2025-07-09 inclusive")
ax.set_yticks([0, 1], ["Internal train", "Internal validation"])
ax.set_xlabel("Calendar date (official training rows only)")
ax.set_title("Internal temporal validation — frozen chronology")
ax.legend(loc="best", fontsize=8)
save_figure(fig, "F13_internal_temporal_split_timeline.png")
fig, ax = plt.subplots(figsize=(6.5, 3.8))
bars = ax.bar(["Internal train", "Internal validation"], split_table["prevalence"], color=COLORS[1:3])
ax.bar_label(bars, labels=[f"{p:.1%}" for p in split_table["prevalence"]])
ax.set_ylim(0, 1); ax.set_ylabel("Phishing prevalence")
ax.set_title("Internal temporal validation — class prevalence")
save_figure(fig, "F14_internal_split_class_prevalence.png")
print("PASS: F13–F14 saved (full frozen-partition diagnostics).")


In [ ]:
# Cell 6 — Metrics, leakage-safe pipelines, fit instrumentation, synthetic checks
GRIDS = {"M1": [{"C": c} for c in [0.1, 1.0, 10.0]],
         "M2": [{"C": c} for c in [0.1, 1.0, 10.0]],
         "M3": [{"max_depth": d} for d in [None, 20]],
         "M4": [{"max_depth": d} for d in [None, 20]]}
TIE_POLICY = "Maximum internal-validation average_precision_score; exact ties retain first candidate in declared order"
def make_pipeline(mid, params):
    if mid == "M0":
        return Pipeline([("model", DummyClassifier(strategy="most_frequent", random_state=SEED))])
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if mid in ["M1", "M2"]:
        steps += [("scaler", StandardScaler()), ("model", LogisticRegression(
            penalty="l2", solver="lbfgs", max_iter=5000, random_state=SEED, C=params["C"]))]
    else:
        steps += [("model", RandomForestClassifier(n_estimators=300, max_depth=params["max_depth"],
                    random_state=SEED, n_jobs=-1))]
    return Pipeline(steps)
def get_x(mid, ix):
    if mid == "M0": return pd.DataFrame({"dummy_constant": np.zeros(len(ix), dtype=np.int8)})
    return df.loc[df.index[ix], FEATURES[mid]]
def positive_probability(model, x):
    classes = np.asarray(model.classes_)
    col = np.flatnonzero(classes == 1)
    require(len(col) == 1, f"Positive label 1 missing or ambiguous: {classes}")
    p = model.predict_proba(x)[:, int(col[0])]
    require(np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all(), "Invalid positive probabilities")
    return p

def metrics(y, pred, p):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    both = len(np.unique(y)) == 2
    return {"accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
      "recall": recall_score(y, pred, zero_division=0), "f1": f1_score(y, pred, zero_division=0),
      "mcc": matthews_corrcoef(y, pred), "roc_auc": roc_auc_score(y, p) if both else None,
      "AP": average_precision_score(y, p) if both else None,
      "FPR": float(fp / (fp + tn)) if fp + tn else None,
      "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
      "validation_negatives": int(tn + fp), "validation_positives": int(tp + fn),
      "threshold": 0.5, "prediction_rule": "estimator.predict; exact tie goes to class 0"}

def timed_fit(model, x, y, mid, stage, candidate):
    if mid != "M0": require(not x.isna().all().any(), f"{mid}: all-missing fit column")
    before = psutil.Process().memory_info().rss
    start = time.perf_counter()
    print(f"Fitting {stage} {mid} {candidate}: {len(y):,} rows", flush=True)
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(x, y)
    except MemoryError as exc:
        raise RuntimeError("STOP: memory exhausted. Keep approved 300 trees/grid; report the resource blocker.") from exc
    elapsed = time.perf_counter() - start
    messages = [f"{w.category.__name__}: {w.message}" for w in caught]
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
    if not converged: blockers.append(f"Convergence warning: {stage} {mid} {candidate}; review before NB4")
    row = {"model_id": mid, "stage": stage, "candidate": candidate, "fit_rows": len(y),
           "fit_seconds": elapsed, "rss_before_bytes": before,
           "rss_after_bytes": psutil.Process().memory_info().rss,
           "converged": converged, "warnings": " | ".join(messages)}
    runtime_rows.append(row)
    pd.DataFrame(runtime_rows).to_csv(OUT / "logs/nb03_runtime.csv", index=False)
    print(f"Completed in {elapsed:.2f}s; warnings={len(messages)}", flush=True)
    for msg in messages: print(msg)
    return row

# Known-answer metric check; missing-value fit leakage check; reversed class order.
toy_y = np.array([0, 0, 1, 1]); toy_p = np.array([.1, .8, .4, .9])
toy_m = metrics(toy_y, (toy_p > .5).astype(int), toy_p)
require(np.isclose(toy_m["AP"], 5/6) and toy_m["TN"] == toy_m["FP"] == toy_m["FN"] == toy_m["TP"] == 1,
        "Synthetic metric test failed")
class ReversedClasses:
    classes_ = np.array([1, 0])
    def predict_proba(self, x): return np.array([[.8, .2]])
require(np.allclose(positive_probability(ReversedClasses(), [0]), [.8]), "Positive-column lookup failed")
toy_x = pd.DataFrame({"x": [0., 2., np.nan, 4.], "z": [1., 0., 1., 0.]})
toy_pipe = make_pipeline("M1", {"C": 1.0}).fit(toy_x, toy_y)
require(np.isclose(toy_pipe.named_steps["imputer"].statistics_[0], 2.), "Median fit test failed")
toy_pipe.predict(pd.DataFrame({"x": [10000., np.nan], "z": [0., 1.]}))
require(np.isclose(toy_pipe.named_steps["imputer"].statistics_[0], 2.), "Validation changed fitted median")
del toy_pipe

candidate_predictions = {}
def run_candidates(mid, grid):
    x_train, x_val = get_x(mid, train_ix), get_x(mid, val_ix)
    for order, params in enumerate(grid):
        cid = f"{mid}_candidate_{order + 1}"
        model = make_pipeline(mid, params)
        runtime = timed_fit(model, x_train, y_train, mid, "selection", cid)
        p, pred = positive_probability(model, x_val), model.predict(x_val)
        score = metrics(y_val, pred, p)
        require(score["AP"] is not None and np.isfinite(score["AP"]), "AP unavailable for selection")
        candidate_rows.append({"model_id": mid, "candidate_id": cid, "candidate_order": order,
            "parameters": json.dumps(params), "fit_seconds": runtime["fit_seconds"],
            "converged": runtime["converged"], **score})
        candidate_predictions[cid] = (p, pred)
        pd.DataFrame(candidate_rows).to_csv(OUT / "tables/model_candidate_results.csv", index=False)
        print(f"{cid}: internal temporal validation AP={score['AP']:.6f}", flush=True)
        del model
        gc.collect()
    del x_train, x_val
print("PASS: synthetic metric, positive-class lookup, and train-only imputation checks.")


In [ ]:
# Cell 7 — M0 validation baseline fitted ONLY on internal_train
run_candidates("M0", [{}])


In [ ]:
# Cell 8 — M1/M2 L2 LR; train-only median imputation and standardization
for mid in ["M1", "M2"]:
    run_candidates(mid, GRIDS[mid])


In [ ]:
# Cell 9 — M3/M4 RF; sequential 300-tree candidates, train-only imputation
# Four RF selection fits (two depths for each of M3/M4), then two refits in Cell 12.
for mid in ["M3", "M4"]:
    run_candidates(mid, GRIDS[mid])


In [ ]:
# Cell 10 — AP-only selection; retain first predeclared candidate on exact ties
candidate_table = pd.DataFrame(candidate_rows)
require(len(candidate_table) == 11, "Expected 1 dummy + 6 LR + 4 RF candidate results")
selected_rows, validation_rows = [], []
for mid in MODEL_NAMES:
    rows = [r for r in candidate_rows if r["model_id"] == mid]
    # Python max returns the first item when scores are exactly equal.
    best = max(rows, key=lambda r: r["AP"])
    chosen[mid] = json.loads(best["parameters"])
    validation_predictions[mid] = candidate_predictions[best["candidate_id"]]
    selected_rows.append({"model_id": mid, "model_name": MODEL_NAMES[mid],
        "candidate_id": best["candidate_id"], "selected_parameters": best["parameters"],
        "validation_AP": best["AP"], "criterion": "average_precision_score", "tie_policy": TIE_POLICY})
    validation_rows.append({"model_id": mid, "model_name": MODEL_NAMES[mid],
        "evaluation": "internal temporal validation", "mode": RUN_MODE,
        **metrics(y_val, validation_predictions[mid][1], validation_predictions[mid][0])})
selected_table, validation_table = pd.DataFrame(selected_rows), pd.DataFrame(validation_rows)
selected_table.to_csv(OUT / "tables/selected_hyperparameters.csv", index=False)
validation_table.to_csv(OUT / "tables/validation_metrics.csv", index=False)
# Retain identifiers/probabilities from selection-time models, not refitted models.
prediction_table = df.iloc[val_ix][["sha256", "record_id", "date", "label"]].reset_index(drop=True)
for mid, (p, pred) in validation_predictions.items():
    prediction_table[f"{mid}_probability"] = p
    prediction_table[f"{mid}_prediction"] = pred
prediction_table.to_parquet(OUT / "tables/internal_validation_predictions.parquet", index=False)
candidate_predictions.clear(); gc.collect()
print(validation_table[["model_id", "AP", "roc_auc", "f1", "mcc"]].to_string(index=False))
print("Development metrics only. Selected using AP only; exact ties keep first candidate.")


In [ ]:
# Cell 11 — Selected internal-validation model comparisons; M0–M4 order
suffix = " [SMOKE: provisional]" if RUN_MODE == "SMOKE" else ""
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(list(MODEL_NAMES), validation_table["AP"], color=COLORS)
ax.bar_label(bars, fmt="%.3f", fontsize=9)
ax.set_ylim(0, 1.08); ax.set_ylabel("Average Precision (sklearn AP)")
ax.set_title("Internal temporal validation — AP" + suffix)
save_figure(fig, "F15_validation_model_AP_comparison.png")
for curve in ["PR", "ROC"]:
    fig, ax = plt.subplots(figsize=(6.5, 5))
    for i, mid in enumerate(MODEL_NAMES):
        p = validation_predictions[mid][0]
        if curve == "PR":
            precision, recall, _ = precision_recall_curve(y_val, p)
            ax.step(recall, precision, where="post", color=COLORS[i], label=f"{mid}: {MODEL_NAMES[mid]}")
        else:
            fpr, tpr, _ = roc_curve(y_val, p)
            ax.plot(fpr, tpr, color=COLORS[i], label=f"{mid}: {MODEL_NAMES[mid]}")
    if curve == "PR":
        ax.axhline(y_val.mean(), color="black", linestyle=":", label="Validation prevalence")
        ax.set(xlabel="Recall", ylabel="Precision")
    else:
        ax.plot([0, 1], [0, 1], "k:", label="Chance reference")
        ax.set(xlabel="False positive rate", ylabel="True positive rate")
    ax.set(xlim=(0, 1), ylim=(0, 1.02), title="Internal temporal validation — " + curve + suffix)
    ax.legend(fontsize=8, loc="best")
    save_figure(fig, f"F{'16' if curve == 'PR' else '17'}_validation_{curve}_curves.png")
# M0 has constant scores: its curve is degenerate and AP equals prevalence.
fig, axes = plt.subplots(2, 3, figsize=(10, 6.5))
for ax, mid in zip(axes.flat, MODEL_NAMES):
    cm = confusion_matrix(y_val, validation_predictions[mid][1], labels=[0, 1])
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=len(y_val))
    for (r, c), value in np.ndenumerate(cm):
        name = [["TN", "FP"], ["FN", "TP"]][r][c]
        ax.text(c, r, f"{name}\n{value:,}", ha="center", va="center", color="black")
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Benign", "Phishing"],
           yticklabels=["Benign", "Phishing"], xlabel="Predicted", ylabel="Actual", title=mid)
axes.flat[-1].axis("off")
fig.suptitle("Internal temporal validation — absolute confusion counts" + suffix)
save_figure(fig, "F18_validation_confusion_matrices.png")
print("PASS: F15–F18 saved; AP is not trapezoidal PR area; M0 is a constant-score baseline.")


In [ ]:
# Cell 12 — Freeze BEFORE refitting; all official-training rows in FULL
MODEL_FILES = {"M0": "M0_dummy_majority.joblib", "M1": "M1_lr_url.joblib",
 "M2": "M2_lr_hybrid.joblib", "M3": "M3_rf_url.joblib", "M4": "M4_rf_hybrid.joblib"}
config = {"run_id": RUN_ID, "mode": RUN_MODE, "dataset": DATASET, "input_provenance": input_records,
    "temporal_split": split_info, "candidate_grids": GRIDS, "chosen_parameters": chosen,
    "selection_metric": "internal_validation average_precision_score", "tie_policy": TIE_POLICY,
    "positive_label": 1, "threshold": .5, "prediction_rule": "estimator.predict; tie -> class 0",
    "seed": SEED, "packages": packages, "official_test_loaded": False,
    "official_test_evaluated": False, "scope_changes": [], "config_frozen": True,
    "refit_scope": "all official train" if RUN_MODE == "FULL" else "SMOKE sampled official train only",
    "refit_rows": len(refit_ix), "models": {}}
for mid in MODEL_NAMES:
    example = make_pipeline(mid, chosen[mid])
    config["models"][mid] = {"role": MODEL_NAMES[mid], "features": FEATURES[mid],
       "estimator_class": type(example.named_steps["model"]).__name__,
       "estimator_parameters": example.named_steps["model"].get_params(deep=False),
       "preprocessing": {name: step.get_params(deep=False) for name, step in example.steps[:-1]},
       "artifact": "models/" + MODEL_FILES[mid]}
    del example
write_json(OUT / "manifests/frozen_model_config.json", config)
CONFIG_HASH = sha256_file(OUT / "manifests/frozen_model_config.json")
require(RUN_MODE != "FULL" or len(refit_ix) == EXPECTED_ROWS, "FULL refit must use every official training row")
probe_ix = refit_ix[:min(64, len(refit_ix))]
reload_reference, final_fit_rows = {}, []
for mid in MODEL_NAMES:
    x = get_x(mid, refit_ix)
    model = make_pipeline(mid, chosen[mid])
    fit_record = timed_fit(model, x, y_refit, mid, "final_refit", "frozen")
    final_fit_rows.append(fit_record)
    require(list(model.feature_names_in_) == FEATURES[mid], f"{mid} feature order changed")
    # Standard sklearn pipelines only: joblib loads in NB4 without notebook custom classes.
    model.nb03_provenance_ = {"model_id": mid, "mode": RUN_MODE, "run_id": RUN_ID,
        "fit_rows": len(refit_ix), "fit_scope": config["refit_scope"],
        "features": FEATURES[mid], "config_sha256": CONFIG_HASH,
        "official_test_loaded": False, "official_test_evaluated": False}
    reload_reference[mid] = (positive_probability(model, get_x(mid, probe_ix)), model.predict(get_x(mid, probe_ix)))
    require(shutil.disk_usage(OUT).free > 512 * 2**20, "Low disk before model save; free space and rerun")
    destination = OUT / "models" / MODEL_FILES[mid]
    tmp = destination.with_suffix(".joblib.tmp")
    joblib.dump(model, tmp, compress=3)
    tmp.replace(destination)
    print("Saved", destination.name, f"({destination.stat().st_size / 2**20:.1f} MiB)")
    del model, x
    gc.collect()
print("PASS: frozen models refitted and saved sequentially. M0 also refitted on final scope.")


In [ ]:
# Cell 13 — Sequential reload, schema, fitted preprocessing, prediction equivalence
size_rows, preprocessing_rows = [], []
for mid, name in MODEL_FILES.items():
    path = OUT / "models" / name
    model = joblib.load(path)  # Only load models this notebook just produced.
    require(list(model.feature_names_in_) == FEATURES[mid], f"{mid} reloaded feature order mismatch")
    require(model.nb03_provenance_["fit_rows"] == len(refit_ix), f"{mid} refit row mismatch")
    require(model.nb03_provenance_["run_id"] == RUN_ID, f"{mid} stale model artifact")
    require(model.nb03_provenance_["config_sha256"] == CONFIG_HASH, f"{mid} frozen config mismatch")
    p, pred = positive_probability(model, get_x(mid, probe_ix)), model.predict(get_x(mid, probe_ix))
    require(np.allclose(p, reload_reference[mid][0], rtol=0, atol=1e-12) and
            np.array_equal(pred, reload_reference[mid][1]), f"{mid} serialization predictions differ")
    if mid != "M0":
        imputer = model.named_steps["imputer"]
        require(len(imputer.statistics_) == len(FEATURES[mid]) and np.isfinite(imputer.statistics_).all(),
                f"{mid} missing/dropped imputation statistics")
        scaler = model.named_steps.get("scaler")
        if scaler is not None:
            require(np.all(np.asarray(scaler.n_samples_seen_) == len(refit_ix)), f"{mid} scaler refit scope mismatch")
        for j, feature in enumerate(FEATURES[mid]):
            preprocessing_rows.append({"model_id": mid, "feature": feature,
                "imputation_median": float(imputer.statistics_[j]),
                "scaler_mean": float(scaler.mean_[j]) if scaler is not None else None,
                "scaler_scale": float(scaler.scale_[j]) if scaler is not None else None})
    size_rows.append({"model_id": mid, **file_record(path), "reload_verified": True,
                      "fit_rows": len(refit_ix), "feature_order_verified": True})
    del model
    gc.collect()
runtime_table = pd.DataFrame(runtime_rows)
size_table = pd.DataFrame(size_rows)
preprocessing_table = pd.DataFrame(preprocessing_rows)
runtime_table.to_csv(OUT / "tables/model_training_runtime.csv", index=False)
size_table.to_csv(OUT / "tables/model_file_sizes.csv", index=False)
preprocessing_table.to_csv(OUT / "tables/fitted_preprocessing_parameters.csv", index=False)
print("PASS: all five models reloaded, feature order and predictions matched; refit scope verified.")


In [ ]:
# Cell 14 — Compact workbook, model-selection report, human-readable checks
checklist = pd.DataFrame([
    ("Verified FULL NB2 inputs and exact hashes", True),
    ("Official test loaded", False), ("Official test evaluated", False),
    ("Stored frozen partition agrees with cutoff", True),
    ("Preprocessing fitted inside each candidate pipeline", True),
    ("Selection by internal-validation AP only", True),
    ("Threshold tuning performed", False), ("Predictors exactly 15 URL + 10 HTML", True),
    ("Five final model reload checks passed", len(size_rows) == 5),
    ("Final refit uses all official training rows", RUN_MODE == "FULL" and len(refit_ix) == EXPECTED_ROWS),
    ("Convergence blockers absent", not blockers)], columns=["check", "value"])
setup_table = pd.DataFrame([{"model_id": k, "role": v, "predictor_count": 0 if k == "M0" else len(FEATURES[k]),
                            "mode": RUN_MODE, "evaluation": "internal temporal validation"} for k, v in MODEL_NAMES.items()])
workbook = OUT / "tables/NB03_Temporal_Validation_and_Model_Selection.xlsx"
sheets = {"Research Setup": setup_table, "Frozen Features": feature_table,
    "Temporal Split": split_table, "Class Prevalence": split_table[["partition", "rows", "benign", "phishing", "prevalence"]],
    "M0 Baseline": validation_table[validation_table.model_id == "M0"],
    "Candidate AP Results": candidate_table, "Selected Hyperparameters": selected_table,
    "Internal Validation Metrics": validation_table, "Runtime": runtime_table,
    "Model File Sizes": size_table, "Package Versions": package_table,
    "Leakage Test Embargo": checklist, "Fitted Preprocessing": preprocessing_table}
with pd.ExcelWriter(workbook, engine="openpyxl") as writer:
    for name, table in sheets.items():
        table.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
        for column in ws.columns:
            letter = column[0].column_letter
            ws.column_dimensions[letter].width = min(48, max(14, max(len(str(c.value or "")) for c in column[:30]) + 2))
        for cell in ws[1]: cell.font = openpyxl.styles.Font(bold=True, color="FFFFFF"); cell.fill = openpyxl.styles.PatternFill("solid", fgColor="24445C")
with pd.ExcelFile(workbook) as check:
    require(set(check.sheet_names) == set(sheets), "Workbook sheet verification failed")
report = ["# Notebook 3 model-selection report", f"Mode: {RUN_MODE}",
    "Internal temporal validation only; these are not final official-test results.",
    "AP is sklearn average_precision_score, not trapezoidal precision-recall area.",
    TIE_POLICY, "No threshold optimization; predict() uses a nominal 0.5 boundary, ties to class 0.",
    f"Refit scope: {config['refit_scope']}; rows: {len(refit_ix):,}.",
    "M0 validation fit used internal_train; saved M0 uses the final refit scope.",
    "No test dataset/schema loaded or evaluated; official boundary caveat is retained from the handoff.",
    "All RF fits kept 300 trees; model objects were released between fits.",
    "RSS before/after are snapshots, not measured peak memory. Times are real wall-clock fit times.",
    "Small SMOKE RF scores are provisional code checks, not thesis findings.",
    "Selected settings:"]
report += [f"- {r['model_id']}: {r['selected_parameters']}; validation AP={r['validation_AP']:.6f}" for r in selected_rows]
report += [f"Blockers: {blockers}", "NB4 requires FULL models, frozen_model_config.json and manifest_nb03.json.",
           "Preserve private Kaggle outputs; do not commit large joblib models to the public repository."]
(OUT / "model_selection_report.md").write_text("\n\n".join(report) + "\n", encoding="utf-8")
print("PASS: compact XLSX, CSVs, and model-selection report saved.")


In [ ]:
# Cell 15 — Complete output integrity manifest, readiness gate, optional ZIP (no Cell 16)
require(sha256_file(OUT / "manifests/frozen_model_config.json") == CONFIG_HASH,
        "Frozen configuration changed during refit")
required = ["models/" + name for name in MODEL_FILES.values()] + [
 "tables/NB03_Temporal_Validation_and_Model_Selection.xlsx", "tables/temporal_split_summary.csv",
 "tables/validation_metrics.csv", "tables/model_candidate_results.csv", "tables/selected_hyperparameters.csv",
 "tables/model_training_runtime.csv", "tables/model_file_sizes.csv", "manifests/temporal_split.json",
 "manifests/frozen_model_config.json", "logs/nb03_runtime.csv", "manifests/package_versions_nb03.csv"] + figure_paths
require(len(figure_paths) == 6 and len(set(figure_paths)) == 6, "Expected six named figures F13–F18")
for rel in required:
    require((OUT / rel).is_file() and (OUT / rel).stat().st_size > 0, f"Missing/empty output {rel}")
require(list(validation_table.model_id) == list(MODEL_NAMES), "Missing model validation metrics")
require(len(final_fit_rows) == 5 and all(r["fit_rows"] == len(refit_ix) for r in final_fit_rows), "Final refits incomplete")
require(len(size_rows) == 5 and all(r["reload_verified"] for r in size_rows), "Reload checks incomplete")
# Include only artifacts created by this execution; refuse stray temp model files.
require(not list((OUT / "models").glob("*.tmp")), "Partial model saves present; restart this run")
outputs = {}
for path in sorted(OUT.rglob("*")):
    if path.is_file() and path.name != "manifest_nb03.json" and path.suffix not in {".tmp", ".zip"}:
        outputs[str(path.relative_to(OUT))] = file_record(path)
# Re-hash source inputs at the end to detect accidental modification during execution.
for name, path in inputs.items():
    require(sha256_file(path) == input_records[name]["sha256"], f"Input changed during run: {name}")
ready = RUN_MODE == "FULL" and not blockers and len(refit_ix) == EXPECTED_ROWS
manifest = {"notebook": "03_temporal_validation_and_model_selection.ipynb", "physical_cells": 15,
    "run_id": RUN_ID, "mode": RUN_MODE, "packages": packages, "dataset": DATASET,
    "inputs": input_records, "artifacts": outputs, "temporal_split": split_info,
    "features": {"url": U, "retained_html": H, "combined": F},
    "candidate_grids": GRIDS, "selection_decisions": selected_rows,
    "validation_metrics": validation_rows, "model_fits": final_fit_rows,
    "model_file_sizes": size_rows, "runtime": runtime_rows,
    "elapsed_seconds_before_zip": time.perf_counter() - STARTED,
    "figures": figure_paths, "tables": [p for p in outputs if p.startswith("tables/")],
    "config_frozen": True, "config_sha256": CONFIG_HASH,
    "official_test_loaded": False, "official_test_evaluated": False,
    "blockers": blockers, "ready_for_nb04": ready,
    "manifest_hash_policy": "Manifest excludes itself; optional ZIP is external and excludes itself."}
for rel, record in outputs.items():
    require(sha256_file(OUT / rel) == record["sha256"], f"Output hash verification failed: {rel}")
write_json(OUT / "manifests/manifest_nb03.json", manifest)
print(f"Mode: {RUN_MODE}\nOfficial train rows: {len(df)}")
print(f"Internal train: {len(indices['internal_train'])}\nInternal validation: {len(indices['internal_validation'])}")
print(f"URL features: {len(U)}\nRetained HTML features: {len(H)}")
print("Fitted final models:", ", ".join(MODEL_NAMES))
print("Primary selection metric: validation Average Precision")
print("Official test loaded: False\nOfficial test evaluated: False")
print("Blockers:", blockers, "\nready_for_nb04:", ready)
if blockers:
    print("STOP — resolve recorded blockers before Notebook 4.")
elif ready:
    print("FULL PASS — save the Kaggle version with outputs.")
else:
    print("SMOKE PASS — provisional only; ready_for_nb04=False. Restart and Run All in FULL.")

# Optional packaging at the bottom of the fifteenth physical cell.
if MAKE_ZIP:
    zip_path = OUT.parent / f"NB03_{RUN_MODE}_ALL_OUTPUTS.zip"
    total = sum((OUT / rel).stat().st_size for rel in outputs) + (OUT / "manifests/manifest_nb03.json").stat().st_size
    if shutil.disk_usage(OUT).free < total * 1.1 + 512 * 2**20:
        print("ZIP skipped: insufficient free disk; keep individual outputs (readiness unaffected).")
    else:
        tmp_zip = zip_path.with_suffix(".zip.tmp")
        try:
            with zipfile.ZipFile(tmp_zip, "w", zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as z:
                for rel in list(outputs) + ["manifests/manifest_nb03.json"]:
                    z.write(OUT / rel, arcname=rel)
            with zipfile.ZipFile(tmp_zip) as z:
                require(z.testzip() is None, "ZIP integrity check failed")
            tmp_zip.replace(zip_path)
            print("ZIP saved:", zip_path, "bytes:", zip_path.stat().st_size)
        except OSError as exc:
            tmp_zip.unlink(missing_ok=True)
            print("Optional ZIP failed; individual verified outputs remain:", exc)
print("Output folder:", OUT)
